![DB Academy](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/common/db-academy.png)

# 02 - REQUIRED - Course Setup and Authentication

## Overview

This notebook prepares your lab environment for the **Automated Deployment with Declarative Automation Bundles (DABs)** course. You'll create the dev, stage, and prod catalogs and data the rest of the course depends on, then install and authenticate the **Databricks CLI** so you can work with DABs from inside a notebook.

The CLI authentication pattern shown here is for **training convenience only**. It uses the short-lived API token that Databricks issues to the running notebook session. In a real environment, follow your organization's security policies. 

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Configure your lab environment** by creating the dev, stage, and prod catalogs, schemas, and volumes used throughout the course.
2. **Reference your unique catalogs** using the `catalog_dev`, `catalog_stage`, and `catalog_prod` Python variables.
3. **Authenticate the Databricks CLI** by exporting the notebook's short-lived API token as `DATABRICKS_TOKEN` and the workspace URL as `DATABRICKS_HOST` (training-only pattern).
4. **Install the Databricks CLI** inside a Databricks notebook.
5. **Run basic CLI commands** (`-v`, `--help`, `catalogs list`) to verify the installation and explore the CLI.

**Note**: If you see the warning `Warn: Failed to load git info from /api/2.0/workspace/get-status` while **validating**, **deploying**, or **running** a Declarative Automation Bundle (DAB) during this course, you can safely ignore it and proceed.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select All-Purpose Compute</strong>
  <div style="color:#333;">

This notebook requires **all-purpose compute** (Dedicated). Serverless is not supported for this notebook.

Follow these steps to attach an all-purpose compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your `labuser_USERNAME` cluster.
    - By default, the notebook might use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

⚠️ **NOTE:** If the cluster shows a **terminated** state (red dot in the cluster picker), it needs to be started before you can attach. Click the cluster, then **Start**, and wait a few minutes until you see a green dot.
  </div>
</div>



## A. Classroom Setup

Run the following cell to configure your working environment for this course. 

It will create the following catalogs, volumes and files:
- Catalog: **labuser_UNIQUE_ID_1_dev**
  - Schema: **default**
    - Volume: **health**
      - *dev_health.csv* : Small subset of prod data, anonymized *PII*, 7,500 rows

- Catalog: **labuser_UNIQUE_ID_2_stage**
  - Schema: **default**
    - Volume: **health**
      - *stage_health.csv* : Subset of prod data, 35,000 rows

- Catalog: **labuser_UNIQUE_ID_3_prod**
  - Schema: **default**
    - Volume: **health**
      - *2025-01-01_health.csv*
      - *2025-01-02_health.csv*
      - *2025-01-03_health.csv*
      - Simulates production CSV files landing to this cloud storage location daily.

In [0]:
%run ./Includes/Classroom-Setup-02-catalog-setup-REQUIRED

## B. Explore Your Environment

1. Let's quickly explore the course folder structure and files.

    a. In the left navigation bar, select the folder icon.

    b. Ensure you are in the main course folder named **Automated Deployment with Declarative Automation Bundles**.
    
    c. In the main course folder, you will see various folders, each folder contains specific files for each demonstration or lab.

2. Manually view your catalogs and data for this course.

    a. In the workspace sidebar, select the catalog icon.

    b. Confirm the classroom setup script has created three new catalogs for you:
      - **labuser_UNIQUE_ID_1_dev**
      - **labuser_UNIQUE_ID_2_stage**
      - **labuser_UNIQUE_ID_3_prod**

    c. Expand each **labuser_UNIQUE_ID** catalog and **default** from above and notice the following:
      - A volume named **health** was created in each catalog
      - It contains **one or more CSV files** for the specific environment:
        - development (1 csv file, small sample of production data, no PII)
        - stage (1 csv file, larger sample of production data, contains PII)
        - production (3 csv files)

3. Throughout the course, the following Python variables will be used to dynamically reference your unique course catalogs:
    - `catalog_dev`
    - `catalog_stage`
    - `catalog_prod`

    Run the code below and confirm the variables refer to your catalog names.


In [0]:
print(f'catalog_dev: {catalog_dev}')
print(f'catalog_stage: {catalog_stage}')
print(f'catalog_prod: {catalog_prod}')

## C. Configure Databricks CLI Authentication

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Training Environment Setup</strong>
  <div style="color:#333;">

In this section, you will **install and configure the Databricks CLI** for this course by reading the notebook's short-lived API token and exporting it as `DATABRICKS_TOKEN`, along with the workspace URL as `DATABRICKS_HOST` using the provided classroom setup script.

⚠️ **Important**
- This pattern is for **training convenience only**, not a production pattern.
- The token lives in this notebook's environment variables. Anyone who can run cells in this notebook can read those values.
- The token is **short-lived**. If the notebook detaches or you start a new session, re-run this cell to refresh the credentials.

**In a production environment, follow your organization's security policies. Common patterns:**
  - **Databricks Secret Management** for storing personal access tokens.
  - **OAuth machine-to-machine** with a service principal for automation and CI/CD.
  - **Workload identity federation** for long-running jobs.

**Generate a production-grade token:**
  - **Personal access tokens (PATs)**, for individual user automation:
    [AWS](https://docs.databricks.com/aws/en/dev-tools/auth/pat) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/auth/pat) |
    [GCP](https://docs.databricks.com/gcp/en/dev-tools/auth/pat)
  - **OAuth machine-to-machine (service principal)**, for jobs and CI/CD pipelines:
    [AWS](https://docs.databricks.com/aws/en/dev-tools/auth/oauth-m2m) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/auth/oauth-m2m) |
    [GCP](https://docs.databricks.com/gcp/en/dev-tools/auth/oauth-m2m)
  - **Databricks authentication overview**, full list of supported auth methods:
    [AWS](https://docs.databricks.com/aws/en/dev-tools/auth/) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/auth/) |
    [GCP](https://docs.databricks.com/gcp/en/dev-tools/auth/)
  - **Secret management**, for storing tokens securely once issued:
    [AWS](https://docs.databricks.com/aws/en/security/secrets/) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/security/secrets/) |
    [GCP](https://docs.databricks.com/gcp/en/security/secrets/)
  </div>
</div>


### C1. Install the Databricks CLI

A token is just like a username and password. Treat it as sensitive.

- If a token is exposed, **delete it immediately**
- Never share tokens with others

For this training, we will temporarily store credentials in environment variables so we can use the CLI in a notebook for training.


1. Run the cell below to authenticate and install the CLI for use with a Databricks Notebook. 

    For this training, we will install it in our workspace using a script we set up for you (it uses the notebook token and environment variables).

    **NOTE:** For more information on installing the Databricks CLI, view the **Install or update the Databricks CLI** documentation:
    [AWS](https://docs.databricks.com/aws/en/dev-tools/cli/install) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/install) |
    [GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/install)

In [0]:
%run ./Includes/Classroom-Setup-Common-Install-CLI

### C2. Simple Databricks CLI Commands
The following cells run basic CLI commands in a notebook using `%sh`.
  - `%sh`: Allows you to run shell code in your notebook.

This will get you familiar with the CLI in a notebook and confirm it was installed correctly.

1. Run the following cell to view the version of the CLI. You should see that you are using **Databricks CLI v0.298.0**.


    **NOTE:** View the documentation on running shell commands in Databricks notebooks - **Code languages in notebooks**:
    [AWS](https://docs.databricks.com/aws/en/notebooks/notebooks-code#code-languages-in-notebooks) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/notebooks/notebooks-code#code-languages-in-notebooks) |
    [GCP](https://docs.databricks.com/gcp/en/notebooks/notebooks-code#code-languages-in-notebooks)

In [0]:
%sh
databricks -v

2. Run the `--help` command to view documentation for the Databricks CLI.

    **Databricks CLI commands** documentation:
    [AWS](https://docs.databricks.com/aws/en/dev-tools/cli/commands) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/commands) |
    [GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/commands)

In [0]:
%sh
databricks --help

3. View help for commands to manage catalogs.

In [0]:
%sh
databricks catalogs --help

4. View available catalogs using the Databricks CLI.

In [0]:
%sh
databricks catalogs list

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
  color: #333;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">Production Note</strong>

This course uses **dedicated classic compute** for installing and running the Databricks CLI inside a notebook. We chose classic for a few practical reasons:

- **`%sh` is fully supported and predictable.** The CLI install pattern (writing the binary to `~/bin/databricks` or `/usr/local/bin/databricks`) works the same way on every classic cluster.

- **The lab environment is on classic, Dedicated Access mode, Unrestricted policy**, so the install script in this course matches the cluster you're learning on in your Vocareum provided Databricks Academy lab.

- Running in a notebook enables a streamlined and efficient training experience to learn how to use DABs.

⚠️ **In production, you typically do not run `databricks bundle deploy` from a notebook.**

The standard production pattern is to deploy bundles from:

- A **developer's local machine** with the Databricks CLI configured against a workspace, or
- A **CI/CD runner** (GitHub Actions, Azure DevOps, GitLab CI, Jenkins) authenticated as a **service principal** via OAuth machine-to-machine.

Running DAB commands from a notebook enables you to focus on bundle structure and behavior without installing the CLI locally. Once you understand bundles, switch to deploying from your actual dev machine or your team's CI pipeline.
</div>


## Conclusion

In this notebook, you set up everything required for the rest of the course:

1. Created the **dev**, **stage**, and **prod** catalogs and the **health** volumes that hold the course data.
2. Verified that the `catalog_dev`, `catalog_stage`, and `catalog_prod` Python variables reference your unique catalogs.
3. Configured **Databricks CLI authentication** using the notebook's short-lived API token (training-only pattern).
4. Installed the **Databricks CLI** in your notebook environment.
5. Ran foundational CLI commands (`-v`, `--help`, `catalogs list`) to confirm the installation.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>